# Stroke — Leakage-controlled validation

Accuracy, F1, Recall and ROC-AUC are reported for unweighted logistic regression using full and selected features, and for class-balanced logistic regression using selected features. Imputation, feature selection and scaling are learned from the training data only.

Balanced logistic regression is pre-specified for the XAI extension; it is not identified as the paper’s selected model or chosen using these test results.

Evaluation uses the same stratified 80/20 split and held-out observations as replication, rather than an additional independent validation cohort.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    f1_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

file_path = "../../data/stroke/healthcare-dataset-stroke-data.csv"


## Original leakage-controlled data preparation

In [2]:
df = pd.read_csv(file_path)
df = df[df['gender'] != 'Other'].copy()
df['age'] = df['age'].astype(int)

y = df['stroke'].copy()
X_raw = df.drop(columns=['stroke', 'id']).copy()

# Split before imputation, encoding, feature selection, and scaling.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

for part in (X_train_raw, X_test_raw):
    part['bmi_missing'] = part['bmi'].isna().astype(int)

train_bmi_mean = X_train_raw['bmi'].mean()
X_train_raw['bmi'] = X_train_raw['bmi'].fillna(train_bmi_mean)
X_test_raw['bmi'] = X_test_raw['bmi'].fillna(train_bmi_mean)

categorical_cols = [
    'gender', 'smoking_status', 'Residence_type',
    'work_type', 'ever_married'
]
X_train_encoded = pd.get_dummies(
    X_train_raw, columns=categorical_cols, drop_first=False
)
X_test_encoded = pd.get_dummies(
    X_test_raw, columns=categorical_cols, drop_first=False
).reindex(columns=X_train_encoded.columns, fill_value=0)

# Learn the correlation-based feature set from training data only.
training_for_selection = X_train_encoded.copy()
training_for_selection['stroke'] = y_train.to_numpy()
training_correlations = training_for_selection.corr(numeric_only=True)['stroke'].abs()
selected_features = training_correlations[
    (training_correlations > 0.1)
    & (training_correlations.index != 'stroke')
].index.tolist()

X_train_full = X_train_encoded
X_test_full = X_test_encoded
X_train_sel = X_train_encoded[selected_features]
X_test_sel = X_test_encoded[selected_features]


In [3]:
print('Training rows:', len(X_train_full))
print('Test rows:', len(X_test_full))
print('Full features:', X_train_full.shape[1])
print('Training-selected features:', selected_features)


Training rows: 4087
Test rows: 1022
Full features: 21
Training-selected features: ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi_missing', 'ever_married_No', 'ever_married_Yes']


In [4]:
scaler_full = StandardScaler()
X_train_full = pd.DataFrame(
    scaler_full.fit_transform(X_train_full),
    columns=X_train_encoded.columns, index=X_train_encoded.index
)
X_test_full = pd.DataFrame(
    scaler_full.transform(X_test_full),
    columns=X_train_encoded.columns, index=X_test_encoded.index
)

scaler_sel = StandardScaler()
X_train_sel = pd.DataFrame(
    scaler_sel.fit_transform(X_train_sel),
    columns=selected_features, index=X_train_encoded.index
)
X_test_sel = pd.DataFrame(
    scaler_sel.transform(X_test_sel),
    columns=selected_features, index=X_test_encoded.index
)


In [5]:
# Define the evaluation functions.
def evaluate(y_true, y_pred, y_scores):
    """Calculate the held-out classification metrics."""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    return acc, f1, recall, fpr, tpr, roc_auc


## Existing replicated logistic-regression configurations

In [6]:
# The replicated baseline uses unweighted logistic regression.
replicated_model_name = "Logistic Regression"
models_before = {}
models_after = {}
results_before = {}
results_after = {}

for X_train, X_test, models, results in (
    (X_train_full, X_test_full, models_before, results_before),
    (X_train_sel, X_test_sel, models_after, results_after),
):
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    models[replicated_model_name] = lr
    results[replicated_model_name] = evaluate(
        y_test, lr.predict(X_test), lr.predict_proba(X_test)[:, 1]
    )


## Pre-specified balanced model and held-out comparison

In [7]:
# Pre-specified extension estimator, retained from the source notebook.
# This is distinct from the unweighted logistic regression in replication.
selected_model_name = "Logistic Regression"
final_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)
final_model.fit(X_train_sel, y_train)
print("Pre-specified model:", selected_model_name)


Pre-specified model: Logistic Regression


In [8]:
balanced_result = evaluate(
    y_test,
    final_model.predict(X_test_sel),
    final_model.predict_proba(X_test_sel)[:, 1],
)

validation_results = {}
validation_results["Replicated LR — full features"] = results_before[replicated_model_name]
validation_results["Replicated LR — selected features"] = results_after[replicated_model_name]
validation_results["Balanced LR — selected features"] = balanced_result

validation_df = pd.DataFrame([
    {"Configuration": name, "Accuracy": result[0], "F1": result[1],
     "Recall": result[2], "ROC-AUC": result[5]}
    for name, result in validation_results.items()
])
display(validation_df)


,Configuration,Accuracy,F1,Recall,ROC-AUC
0,Replicated LR — full features,0.951076,0.038462,0.02,0.836975
1,Replicated LR — selected features,0.950098,0.037736,0.02,0.833807
2,Balanced LR — selected features,0.754403,0.241692,0.80,0.833807


## Interpretation

Class weighting changes the operating point rather than the underlying data difficulty. The selected balanced logistic-regression model prioritises sensitivity, reaching recall of 0.800 while producing low precision because stroke prevalence is small. The fixed 0.5 threshold is retained for comparability and is not proposed as a clinical decision threshold.
